# Financial sentiment: TF-IDF vs frozen embeddings vs fine-tuned DistilBERT

Runs the three methods from `src/compliance_ai/nlp` on **Financial PhraseBank** (3,448 expert-labelled
financial news sentences) and compares them on the same held-out test set.

**Before you start:** `Runtime` → `Change runtime type` → **T4 GPU** → Save. Then `Runtime` → `Run all`.
The whole notebook takes about 10 minutes on the free GPU.

Why DistilBERT and not FinBERT? FinBERT was trained on this very dataset, so its test scores here would be inflated.

## 1. Get the project

In [ ]:
REPO = "https://github.com/shubhamyad86-ai/compliance-ai.git"   # change if your repo name differs
!git clone -q $REPO project
%cd project
!pip install -q -e . -r requirements.txt -r requirements-nlp.txt
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the runtime to T4 GPU")

## 2. Train and evaluate the three methods

In [ ]:
!python -m compliance_ai.nlp.train --method tfidf_baseline

In [ ]:
!python -m compliance_ai.nlp.train --method frozen_embeddings

In [ ]:
!python -m compliance_ai.nlp.train --method fine_tuned_transformer

## 3. Compare

In [ ]:
import pandas as pd
cols = ["method", "model", "accuracy", "macro_f1", "f1_negative", "f1_neutral", "f1_positive",
        "latency_ms_per_sentence", "train_seconds"]
pd.read_csv("reports/nlp_comparison.csv")[cols].sort_values("macro_f1", ascending=False)

In [ ]:
from IPython.display import Image, display
for m in ["tfidf_baseline", "frozen_embeddings", "fine_tuned_transformer"]:
    display(Image(f"reports/nlp_confusion_{m}.png", width=330))

## 4. Error analysis: what does the best model still get wrong?

In [ ]:
import json
for e in json.load(open("reports/nlp_errors_fine_tuned_transformer.json")):
    print(f"[true {e['label']:>8} | predicted {e['predicted']:>8}] {e['text']}")

## 5. Try it

In [ ]:
import sys; sys.path.insert(0, "src")   # make the project importable in this notebook
from compliance_ai.nlp.predict import classify
for r in classify([
    "The company reported a pre-tax loss and announced layoffs.",
    "Operating profit rose to EUR 13.1 mn from EUR 8.7 mn.",
    "The annual general meeting will be held in Helsinki on 15 March.",
]):
    print(r["label"], r["confidence"], "-", r["text"])

## 6. Download the results

This saves the trained models and reports as `sentiment_results.zip`. On your laptop, unzip it inside the
project folder (it contains `models/sentiment/...` and `reports/...`), then the API's `/nlp/sentiment` endpoint
uses the fine-tuned model automatically.

In [ ]:
!zip -qr sentiment_results.zip models/sentiment reports/nlp_*
from google.colab import files
files.download("sentiment_results.zip")